# Microsoft Fabric – Power BI Artifact Deployment

This notebook deploys the final packaged Power BI artifacts from the `healthcare-artifacts/<version>` build output in a Fabric Lakehouse into the current Microsoft Fabric workspace using the Fabric REST Items API via `sempy.fabric.FabricRestClient`.

| Artifact | Type |
|---|---|
| Care Management Analytics Semantic Model | Semantic Model |
| OMOP Analytics Semantic Model | Semantic Model |
| Patient Outreach Analytics Semantic Model | Semantic Model |
| Care Management Analytics BI Report | Report |
| Patient Outreach Analytics BI Report | Report |

**Deployment order:** Semantic models are deployed first because reports depend on them.

---

### Prerequisites
- Run this notebook inside a Microsoft Fabric Spark session.
- The user or service account running the notebook must have at least Contributor access to the target workspace.
- The final build artifacts must already be uploaded into the deployment Lakehouse under `Files/hds-build-artifacts/healthcare-artifacts/<artifact-version>/...`.


## 1 · Load common deployment configuration

In [ ]:
%run common_deployment_config

In [ ]:
# Using shared build_artifact_name from common_deployment_config
print('✓ Using build_artifact_name and Power BI config from common_deployment_config')

In [ ]:
# PowerBI-specific imports
import time

print("✓ PowerBI-specific imports loaded")

## 2 · PowerBI-specific configuration

Set the PowerBI-specific values below before running the notebook.

| Variable | Description |
|---|---|
| `POWERBI_ENDPOINT` | XMLA endpoint host. Use `api.fabric.microsoft.com` for Fabric, `api.powerbi.com` for Premium. |
| `UPDATE_IF_EXISTS` | When `True`, re-running the notebook updates existing items instead of skipping. |

> **Note:** Common configuration (ARTIFACT_VERSION, ARTIFACT_LAKEHOUSE_NAME, WORKSPACE_ID, etc.) is loaded from `common_deployment_config`.

In [ ]:
# =============================================================================
# POWERBI-SPECIFIC CONFIGURATION
# =============================================================================
# (Common config loaded from common_deployment_config)

# Fabric XMLA / Power BI endpoint host used in report connection strings.
# For Microsoft Fabric: api.fabric.microsoft.com
# For Power BI Premium: api.powerbi.com
POWERBI_ENDPOINT: str = "api.fabric.microsoft.com"

# When True, update an existing item's definition instead of skipping it.
UPDATE_IF_EXISTS: bool = True

print("✓ PowerBI deployer configuration:")
print(f"  Version: {ARTIFACT_VERSION}")
print(f"  Lakehouse: {ARTIFACT_LAKEHOUSE_NAME}")
print(f"  PowerBI Endpoint: {POWERBI_ENDPOINT}")
print(f"  Update if exists: {UPDATE_IF_EXISTS}")

## 3 · Construct PowerBI artifact paths

Build artifact paths using common configuration variables.

In [ ]:
# Build PowerBI-specific artifact paths using BASE_DIST_PATH from common config
BASE_ARTIFACTS_PATH: str = f"{BASE_DIST_PATH}/healthcare-artifacts/{ARTIFACT_VERSION}"

print(f"✓ Workspace ID: {WORKSPACE_ID}")
print(f"✓ Workspace Name: {WORKSPACE_NAME}")
print(f"✓ Endpoint URI: {ENDPOINT_URI}")
print(f"✓ Build artifacts path: {BASE_ARTIFACTS_PATH}")

# Initialize Fabric REST client
_client = FabricRestClient()

## 4 · Fabric API helpers

All REST calls are made through `sempy.fabric.FabricRestClient`, which handles authentication transparently using the Fabric session token — no bearer token management required.

In [ ]:
def _poll_long_running(location: str, retry_after: int = 5, max_wait: int = 300) -> dict:
    """Poll a Fabric long-running operation URL until it completes."""
    # FabricRestClient paths are relative; strip the base URL if present.
    relative = location.replace("https://api.fabric.microsoft.com", "")
    elapsed = 0
    while elapsed < max_wait:
        time.sleep(retry_after)
        elapsed += retry_after
        resp = _client.get(relative)
        resp.raise_for_status()
        state = resp.json()
        status = state.get("status", "").lower()
        if status == "succeeded":
            return state
        if status in ("failed", "cancelled"):
            raise RuntimeError(f"Long-running operation failed: {state}")
        print(f"  ... still running ({elapsed}s elapsed, status={status})")
    raise TimeoutError(f"Long-running operation did not complete within {max_wait}s")


def list_workspace_items(item_type: Optional[str] = None) -> List[dict]:
    """Return all items in the workspace, optionally filtered by type."""
    path = f"/v1/workspaces/{WORKSPACE_ID}/items"
    if item_type:
        path += f"?type={item_type}"
    resp = _client.get(path)
    resp.raise_for_status()
    return resp.json().get("value", [])


def find_item(display_name: str, item_type: str) -> Optional[dict]:
    """Return the first workspace item matching name + type, or None."""
    for item in list_workspace_items(item_type):
        if item.get("displayName") == display_name:
            return item
    return None


def create_item(display_name: str, item_type: str, parts: List[dict]) -> dict:
    """Create or update a Fabric workspace item from base64-encoded definition parts.

    If UPDATE_IF_EXISTS=True and the item already exists its definition is
    updated; otherwise it is created. Returns the resulting item dict.
    """
    existing = find_item(display_name, item_type)

    if existing:
        if not UPDATE_IF_EXISTS:
            print(f"  [skip] '{display_name}' ({item_type}) already exists.")
            return existing
        print(f"  [update] '{display_name}' ({item_type}) – updating definition.")
        item_id = existing["id"]
        path = f"/v1/workspaces/{WORKSPACE_ID}/items/{item_id}/updateDefinition"
        body = {"definition": {"parts": parts}}
        resp = _client.post(path, json=body)
    else:
        print(f"  [create] '{display_name}' ({item_type})")
        path = f"/v1/workspaces/{WORKSPACE_ID}/items"
        body = {
            "displayName": display_name,
            "type": item_type,
            "definition": {"parts": parts},
        }
        resp = _client.post(path, json=body)

    # Handle 202 long-running operations
    if resp.status_code == 202:
        location = resp.headers.get("Location") or resp.headers.get("location", "")
        retry_after = int(resp.headers.get("Retry-After", 5))
        print(f"  [async] Operation accepted – polling ...")
        _poll_long_running(location, retry_after)
        item = find_item(display_name, item_type)
        if not item:
            raise RuntimeError(f"Item '{display_name}' not found after async creation.")
        return item

    if resp.status_code not in (200, 201):
        raise RuntimeError(
            f"Failed to deploy '{display_name}': HTTP {resp.status_code} – {resp.text}"
        )

    if existing:
        return find_item(display_name, item_type) or existing
    return resp.json()


print("✓ Fabric API helper functions defined.")


## 6 · Artifact file helpers

These helpers read the packaged Power BI files from the uploaded `healthcare-artifacts/<version>` structure in the deployment Lakehouse, load the lakehouse manifest, resolve the backing Fabric lakehouses, and substitute semantic model placeholders before deployment.


In [ ]:
BASE = BASE_ARTIFACTS_PATH.rstrip("/")
print(f"[artifacts] Base path: {BASE}")

MANIFEST_PATH = (
    f"{BASE_DIST_PATH}/healthcare-configuration/{ARTIFACT_VERSION}/"
    "system-configurations/LakehouseHydrationManifest.json"
)
print(f"[artifacts] Manifest path: {MANIFEST_PATH}")

MAX_TEXT_FILE_BYTES = 100 * 1024 * 1024


def _read_text(abfss_path: str, max_bytes: int = MAX_TEXT_FILE_BYTES) -> str:
    """Read full text content from OneLake without truncation."""
    from pyspark.sql import SparkSession

    active_spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
    lines = [row.value for row in active_spark.read.text(abfss_path).toLocalIterator()]
    content = "\n".join(lines)

    if not content:
        raise RuntimeError(f"File is empty or unreadable: {abfss_path}")

    size_bytes = len(content.encode("utf-8"))
    if size_bytes > max_bytes:
        raise RuntimeError(
            f"File exceeds configured max read size ({size_bytes} > {max_bytes} bytes): {abfss_path}"
        )

    return content


def b64(abfss_path: str) -> str:
    """Read a text artifact from OneLake and return it as base64."""
    raw = _read_text(abfss_path).encode("utf-8")
    return base64.b64encode(raw).decode("utf-8")


def make_part(relative_path: str, abfss_path: str) -> dict:
    """Build a single Fabric item definition part dict from a OneLake file."""
    return {
        "path": relative_path,
        "payload": b64(abfss_path),
        "payloadType": "InlineBase64",
    }


def make_inline_part(relative_path: str, content: str) -> dict:
    """Build a Fabric item definition part from in-memory text."""
    return {
        "path": relative_path,
        "payload": base64.b64encode(content.encode("utf-8")).decode("utf-8"),
        "payloadType": "InlineBase64",
    }


def _normalize_name(value: str) -> str:
    return value.strip().lower().replace("-", "_").replace(" ", "_")


def _json_string_escape(value: str) -> str:
    """Escape a replacement value for insertion inside an existing JSON string literal."""
    return json.dumps(value)[1:-1]


def _parse_json_or_raise(content: str, label: str) -> Dict[str, Any]:
    """Parse JSON and raise with rich context when invalid."""
    try:
        parsed = json.loads(content)
    except json.JSONDecodeError as ex:
        lines = content.splitlines()
        start = max(1, ex.lineno - 3)
        end = min(len(lines), ex.lineno + 3)
        context = "\n".join(
            f"{i}: {lines[i - 1]}" for i in range(start, end + 1)
        )
        raise RuntimeError(
            f"{label} is invalid JSON at line {ex.lineno}, column {ex.colno}: {ex.msg}\n"
            f"Context:\n{context}"
        ) from ex

    if not isinstance(parsed, dict):
        raise RuntimeError(f"{label} must be a JSON object, got {type(parsed).__name__}.")
    return parsed


def _canonicalize_json(content: str, label: str) -> str:
    """Return canonical JSON text after validating structure."""
    parsed = _parse_json_or_raise(content, label)
    return json.dumps(parsed, ensure_ascii=False, separators=(",", ":"))


def load_lakehouse_manifest() -> Dict[str, Any]:
    """Load the lakehouse manifest from the uploaded build artifacts."""
    content = _read_text(MANIFEST_PATH, 1024 * 1024)
    manifest = json.loads(content)
    if not isinstance(manifest, dict) or not manifest:
        raise ValueError("Lakehouse manifest is empty or invalid")
    print(f"✓ Loaded lakehouse manifest with {len(manifest)} entries")
    return manifest


LAKEHOUSE_MANIFEST = load_lakehouse_manifest()


def resolve_lakehouse_binding(manifest_key: str) -> Dict[str, str]:
    """Resolve a manifest lakehouse key to the actual workspace lakehouse + SQL endpoint."""
    if manifest_key not in LAKEHOUSE_MANIFEST:
        raise KeyError(f"Manifest key not found: {manifest_key}")

    target_names = {
        _normalize_name(manifest_key),
        _normalize_name(manifest_key.replace("-", "_")),
    }

    lakehouses = list_workspace_items("Lakehouse")
    matches: List[dict] = []

    for lakehouse in lakehouses:
        display_name = lakehouse.get("displayName", "")
        normalized_display = _normalize_name(display_name)
        if normalized_display in target_names:
            matches = [lakehouse]
            break
        if any(name in normalized_display for name in target_names):
            matches.append(lakehouse)

    if not matches:
        raise RuntimeError(
            f"Could not resolve a workspace lakehouse for manifest key '{manifest_key}'. "
            f"Available lakehouses: {[item.get('displayName') for item in lakehouses]}"
        )

    chosen = matches[0]
    lakehouse_id = chosen["id"]
    details_response = _client.get(f"/v1/workspaces/{WORKSPACE_ID}/lakehouses/{lakehouse_id}")
    details_response.raise_for_status()
    details = details_response.json()

    sql_props = details.get("properties", {}).get("sqlEndpointProperties", {})
    connection_string = sql_props.get("connectionString")
    sql_endpoint_id = sql_props.get("id")
    if not connection_string or not sql_endpoint_id:
        raise RuntimeError(
            f"Lakehouse '{details.get('displayName')}' does not expose a ready SQL endpoint."
        )

    binding = {
        "manifestKey": manifest_key,
        "lakehouseId": lakehouse_id,
        "lakehouseName": details["displayName"],
        "sqlEndpoint": connection_string,
        "sqlEndpointId": sql_endpoint_id,
    }
    print(
        f"✓ Resolved {manifest_key} -> {binding['lakehouseName']} "
        f"(SQL endpoint id: {binding['sqlEndpointId']})"
    )
    return binding


def render_model_bim(abfss_template_path: str, replacements: Dict[str, str]) -> str:
    """Render and validate a semantic model template by replacing placeholders."""
    content = _read_text(abfss_template_path)
    for placeholder, value in replacements.items():
        content = content.replace(placeholder, _json_string_escape(value))

    unresolved = sorted(set(re.findall(r"%%[A-Za-z0-9_]+%%", content)))
    if unresolved:
        raise RuntimeError(
            "Unresolved placeholders remain in model.bim: " + ", ".join(unresolved)
        )

    # Normalize JSON before upload so Fabric receives canonical, validated JSON.
    return _canonicalize_json(content, f"Rendered model.bim ({abfss_template_path})")


def build_pbir(abfss_template_path: str, semantic_model_name: str, semantic_model_id: str) -> str:
    """Read and validate a definition.pbir template from OneLake."""
    content = _read_text(abfss_template_path)
    content = content.replace("%%powerbi_endpoint%%", _json_string_escape(POWERBI_ENDPOINT))
    content = content.replace("%%workspace_name%%", _json_string_escape(WORKSPACE_NAME))
    content = content.replace("%%semantic_model_name%%", _json_string_escape(semantic_model_name))
    content = content.replace("%%semantic_model_id%%", _json_string_escape(semantic_model_id))
    return _canonicalize_json(content, f"Rendered definition.pbir ({abfss_template_path})")


print("✓ Artifact file helpers defined.")


## 7 · Deployment configuration

Define all semantic models and reports with their configurations in a structured way.

### 7.1 · Semantic Models Configuration

In [ ]:
# SEMANTIC_MODELS now provided by common_deployment_config
print(f"✓ SEMANTIC_MODELS loaded from common_deployment_config: {len(SEMANTIC_MODELS)} model(s)")

### 7.2 · Reports Configuration

In [ ]:
# REPORTS now provided by common_deployment_config
print(f"✓ REPORTS loaded from common_deployment_config: {len(REPORTS)} report(s)")

In [ ]:
# MIGRATION: Rename any existing unprefixed items to the canonical prefixed names
# This enforces Option A: always use prefixed display names. If a prefixed item
# already exists we do not rename or create a duplicate (we skip).
migrated = 0
migration_failed = 0
skipped = 0

print('' + '=' * 80)
print('MIGRATING EXISTING ITEMS TO PREFIXED NAMES')
print('=' * 80 + '')

# Helper to rename a workspace item displayName via the Items API
def _rename_item_to_prefixed(item: dict, prefixed_name: str) -> bool:
    item_id = item.get('id')
    try:
        path = f"/v1/workspaces/{WORKSPACE_ID}/items/{item_id}"
        body = {'displayName': prefixed_name}
        resp = _client.patch(path, json=body)
        resp.raise_for_status()
        return True
    except Exception as ex:  # noqa: BLE001
        print(f"  ❌ Failed to rename item id={item_id}: {ex}")
        return False

# Migrate semantic models (if an unprefixed item exists and no prefixed item exists)
for sm in SEMANTIC_MODELS:
    name = sm['name']
    prefixed = build_artifact_name(name)
    print(f"Processing SemanticModel: '{name}' -> '{prefixed}'")

    if find_item(prefixed, 'SemanticModel') is not None:
        print(f"  [skip] Prefixed SemanticModel '{prefixed}' already exists.")
        skipped += 1
        continue

    raw = find_item(name, 'SemanticModel')
    if raw is None:
        print(f"  (no existing SemanticModel named '{name}' found)")
        continue

    # Rename in-place to the prefixed display name
    print(f"  [migrate] Renaming existing '{name}' (id={raw.get('id')}) to '{prefixed}'")
    ok = _rename_item_to_prefixed(raw, prefixed)
    if ok:
        migrated += 1
    else:
        migration_failed += 1

# Migrate reports with the same policy
for rpt in REPORTS:
    name = rpt['name']
    prefixed = build_artifact_name(name)
    print(f"Processing Report: '{name}' -> '{prefixed}'")

    if find_item(prefixed, 'Report') is not None:
        print(f"  [skip] Prefixed Report '{prefixed}' already exists.")
        skipped += 1
        continue

    raw = find_item(name, 'Report')
    if raw is None:
        print(f"  (no existing Report named '{name}' found)")
        continue

    print(f"  [migrate] Renaming existing '{name}' (id={raw.get('id')}) to '{prefixed}'")
    ok = _rename_item_to_prefixed(raw, prefixed)
    if ok:
        migrated += 1
    else:
        migration_failed += 1

print('' + '=' * 80)
print('MIGRATION SUMMARY')
print('=' * 80)
print(f"  Renamed         : {migrated}")
print(f"  Skipped         : {skipped}")
print(f"  Failed          : {migration_failed}")


In [ ]:
# Deploy all semantic models in a loop (prefix-aware)
semantic_model_ids = {}

sm_deployed = 0
sm_failed = 0
sm_skipped = 0

print("\n" + "=" * 80)
print("DEPLOYING SEMANTIC MODELS")
print("=" * 80 + "\n")

for sm_config in SEMANTIC_MODELS:
    name = sm_config["name"]
    print(f"📊 Deploying: {name}")

    # Build prefixed display name and check both raw and prefixed variants
    prefixed_name = build_artifact_name(name)
    existing_sm = find_item(name, "SemanticModel") or find_item(prefixed_name, "SemanticModel")
    if existing_sm is not None:
        print(f"  [skip] '{existing_sm.get('displayName')}' (SemanticModel) already exists.")
        semantic_model_ids[name] = existing_sm["id"]
        print(f"  ✓ Existing Semantic Model ID: {existing_sm['id']}\n")
        sm_skipped += 1
        continue

    try:
        # Resolve lakehouse binding
        binding = resolve_lakehouse_binding(sm_config["lakehouse_binding"])

        # Build placeholders dictionary with actual values
        replacements = {}
        for placeholder, binding_key in sm_config["placeholders"].items():
            replacements[placeholder] = binding[binding_key]

        # Render model.bim with placeholders
        model_content = render_model_bim(
            f"{sm_config['dir']}/{sm_config['model_file']}",
            replacements,
        )

        # Deploy semantic model using prefixed display name
        sm_item = create_item(
            display_name=prefixed_name,
            item_type="SemanticModel",
            parts=[
                make_part("definition.pbism", f"{sm_config['dir']}/{sm_config['definition_file']}"),
                make_inline_part("model.bim", model_content),
            ],
        )

        # Store the ID for later reference keyed by logical name
        semantic_model_ids[name] = sm_item["id"]
        print(f"  ✓ Semantic Model ID: {sm_item['id']} (displayName={sm_item.get('displayName')})\n")
        sm_deployed += 1

    except Exception as ex:  # noqa: BLE001
        sm_failed += 1
        print(f"  ❌ Failed to deploy semantic model '{name}': {ex}\n")

print("=" * 80)
print("SEMANTIC MODEL DEPLOYMENT SUMMARY")
print("=" * 80)
print(f"  Total configured : {len(SEMANTIC_MODELS)}")
print(f"  Deployed         : {sm_deployed}")
print(f"  Skipped (exists) : {sm_skipped}")
print(f"  Failed           : {sm_failed}")
print("=" * 80 + "\n")


## 9 · Deploy Reports

Deploy all reports using loop-based approach. Each `definition.pbir` references its semantic model via a templated XMLA connection string.

In [ ]:
# Deploy all reports in a loop (prefix-aware)
report_ids = {}

rep_deployed = 0
rep_failed = 0
rep_skipped = 0

print("\n" + "=" * 80)
print("DEPLOYING REPORTS")
print("=" * 80 + "\n")

for report_config in REPORTS:
    name = report_config["name"]
    print(f"📈 Deploying: {name}")

    # Build prefixed report display name
    prefixed_report_name = build_artifact_name(name)

    # Check if report exists under raw or prefixed names
    existing_report = find_item(name, "Report") or find_item(prefixed_report_name, "Report")
    if existing_report is not None:
        print(f"  [skip] '{existing_report.get('displayName')}' (Report) already exists.")
        report_ids[name] = existing_report["id"]
        print(f"  ✓ Existing Report ID: {existing_report['id']}\n")
        rep_skipped += 1
        continue

    # Resolve semantic model (prefer workspace items if present)
    semantic_model_name = report_config["semantic_model_ref"]
    prefixed_sm_name = build_artifact_name(semantic_model_name)
    sm_item = find_item(semantic_model_name, "SemanticModel") or find_item(prefixed_sm_name, "SemanticModel")
    sm_id = sm_item["id"] if sm_item else semantic_model_ids.get(semantic_model_name)
    sm_display = sm_item.get("displayName") if sm_item else (prefixed_sm_name if sm_id else None)

    if not sm_id:
        rep_failed += 1
        print(f"  ❌ Failed: Semantic model '{semantic_model_name}' not found. Cannot deploy report.\n")
        continue

    try:
        # Build PBIR content with semantic model reference (use actual display name)
        pbir_content = build_pbir(
            abfss_template_path=f"{report_config['dir']}/definition.pbir",
            semantic_model_name=sm_display,
            semantic_model_id=sm_id,
        )

        # Deploy report using prefixed display name
        report_item = create_item(
            display_name=prefixed_report_name,
            item_type="Report",
            parts=[
                make_inline_part("definition.pbir", pbir_content),
                make_part("report.json", f"{report_config['dir']}/report.json"),
                make_part(
                    "StaticResources/SharedResources/BaseThemes/" + report_config["theme_file"],
                    f"{report_config['dir']}/{report_config['theme_file']}",
                ),
            ],
        )

        # Store the ID for summary
        report_ids[name] = report_item["id"]
        print(f"  ✓ Report ID: {report_item['id']} (displayName={report_item.get('displayName')})\n")
        rep_deployed += 1

    except Exception as ex:  # noqa: BLE001
        rep_failed += 1
        print(f"  ❌ Failed to deploy report '{name}': {ex}\n")

print("=" * 80)
print("REPORT DEPLOYMENT SUMMARY")
print("=" * 80)
print(f"  Total configured : {len(REPORTS)}")
print(f"  Deployed         : {rep_deployed}")
print(f"  Skipped (exists) : {rep_skipped}")
print(f"  Failed           : {rep_failed}")
print("=" * 80 + "\n")


## 10 · Deployment summary

In [ ]:
# Build summary from deployed items (include deployed displayName)
summary = []

# Helper to fetch an item by id and return its displayName (safe)
def _get_display_name_by_id(item_id: str) -> str:
    try:
        resp = _client.get(f"/v1/workspaces/{WORKSPACE_ID}/items/{item_id}")
        resp.raise_for_status()
        return resp.json().get('displayName')
    except Exception:
        return None

# Add semantic models to summary
for name, item_id in semantic_model_ids.items():
    deployed_name = _get_display_name_by_id(item_id) or build_artifact_name(name)
    summary.append({"Name": name, "DeployedName": deployed_name, "Type": "SemanticModel", "ID": item_id})

# Add reports to summary
for name, item_id in report_ids.items():
    deployed_name = _get_display_name_by_id(item_id) or build_artifact_name(name)
    summary.append({"Name": name, "DeployedName": deployed_name, "Type": "Report", "ID": item_id})

# Display summary
print("\n" + "=" * 120)
print("DEPLOYMENT SUMMARY")
print("=" * 120)
print(f"{'Name':<40} {'DeployedName':<40} {'Type':<15} {'ID'}")
print("-" * 120)
for row in summary:
    print(f"{row['Name']:<40} {row.get('DeployedName',''):<40} {row['Type']:<15} {row['ID']}")
print("=" * 120)
print(f"\nWorkspace     : {WORKSPACE_NAME}")
print(f"Workspace ID  : {WORKSPACE_ID}")
print(f"Fabric portal : https://app.fabric.microsoft.com/groups/{WORKSPACE_ID}")
print(f"Total deployed: {len(summary)} items ({len(semantic_model_ids)} semantic models, {len(report_ids)} reports)")
print("=" * 120 + "\n")

# Display as DataFrame if Spark session is active
from pyspark.sql import SparkSession  # noqa: PLC0415
active_spark = SparkSession.getActiveSession()
if active_spark:
    display(active_spark.createDataFrame(summary))  # noqa: F821